# core

> Fill in a module description here

In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export

import os, logging, configparser

In [ ]:
#| exporti
syslog = logging.getLogger(f"root.{__name__}")


### Configuration

In [ ]:
#| exporti

def get_config_path():
    basepath =os.getenv('APPDATA', os.getenv('HOME', os.getenv('USERPROFILE', os.getcwd())))
    config_path = os.path.join(basepath, '.edtravelcompanion')
    os.makedirs(config_path, exist_ok=True)
    return config_path

def get_log_path():
    log_path = os.path.join(get_config_path(), 'logs')
    os.makedirs(log_path, exist_ok=True)
    return log_path

In [ ]:
print(get_config_path())
print(get_log_path())

C:\Users\fenke\AppData\Roaming\.edtravelcompanion
C:\Users\fenke\AppData\Roaming\.edtravelcompanion\logs


In [ ]:
#| export

def init_configuration(
        config_file = os.path.join(get_config_path(), 'edcompanion.ini') # filename including path for the config file 
):
    
    """
        Opens the specified configuration file (.ini) and 
        returns a proxy-object that saves on modify.
    """

    if not hasattr(init_configuration, 'configurations'):
        init_configuration.configurations = {}

    if config_file in init_configuration.configurations:
        return configuration

    configuration = configparser.ConfigParser()
    
    def update_config():
        with open(config_file, 'w') as f:
            configuration.write(f)

    class _config_section_proxy():
        def __init__(self, section_key):
            self.section_key = section_key

        def __getitem__(self, key):
            return configuration.__getitem__(self.section_key).__getitem__(key)

        def __setitem__(self, key, value):
            configuration.__getitem__(self.section_key).__setitem__(key, str(value))
            update_config()

        def __delitem__(self, key):
            configuration.__getitem__(self.section_key).__delitem__(key)
            update_config()

        def __contains__(self, key):
            return configuration.__getitem__(self.section_key).__contains__(key)

        def __repr__(self):
            return configuration.__getitem__(self.section_key).__repr__()

    class _config_proxy():
        def __init__(self, config_file):
            syslog.info(f"init configuration for {config_file}")
            configuration.read(config_file, encoding='utf-8')

        def __getitem__(self, key):
            return _config_section_proxy(key)

        def __setitem__(self, key, value):
            configuration.__setitem__(key, value)
            update_config()

        def __delitem__(self, key):
            configuration.__delitem__(key)
            update_config()

        def __contains__(self, key):
            return configuration.__contains__(key)

        def __repr__(self):
            return f"Configuration proxy for {config_file} using {configuration.__repr__()}"
    
    init_configuration.configurations[config_file] = _config_proxy(config_file)

    return init_configuration.configurations[config_file]


Create the proxy object with init_configuration and access sections within and key-value pairs in each section

In [ ]:
configration = init_configuration()
print(configration)

Configuration proxy for C:\Users\fenke\AppData\Roaming\.edtravelcompanion\edcompanion.ini using <configparser.ConfigParser object>


In [ ]:
configration["DBCONFIG"] = {
"db-url": "mydb.com/connection-string",
"port": "3306",
"ipaddr": "100.10.10.1"
}


In [ ]:
print(configration["DBCONFIG"]["db-url"])

mydb.com/connection-string


In [ ]:
db_section = configration["DBCONFIG"]
print(db_section["db-url"])

mydb.com/connection-string


In [ ]:
#| hidey
import nbdev; nbdev.nbdev_export()